In [10]:
print(df_master.columns)

NameError: name 'df_master' is not defined

In [ ]:
import pandas as pd

file_path = "../data/raw/01_fund_master.csv"
df_master = pd.read_csv(file_path)

print("==== FUND MASTER SUMMARY ====\n")

# 1. Unique Fund Houses
unique_fund_houses = df_master['fund_house'].unique()
print(f"1. Total Unique Fund Houses ({len(unique_fund_houses)}):")
print(unique_fund_houses)
print("-" * 50)

# 2. Unique Categories
unique_categories = df_master['category'].unique()
print(f"2. Total Unique Categories ({len(unique_categories)}):")
print(unique_categories)
print("-" * 50)

# 3. Unique Sub-categories
unique_sub_categories = df_master['sub_category'].unique()
print(f"3. Total Unique Sub-categories ({len(unique_sub_categories)}):")
print(unique_sub_categories)
print("-" * 50)

# 4. Unique Risk Categories 
unique_risk_categories = df_master['risk_category'].unique()
print(f"4. Total Unique Risk Categories ({len(unique_risk_categories)}):")
print(unique_risk_categories)
print("-" * 50)

==== FUND MASTER SUMMARY ====

1. Total Unique Fund Houses (10):
['SBI Mutual Fund' 'HDFC Mutual Fund' 'ICICI Prudential MF'
 'Nippon India MF' 'Kotak Mahindra MF' 'Axis Mutual Fund'
 'Aditya Birla Sun Life MF' 'UTI Mutual Fund' 'Mirae Asset MF'
 'DSP Mutual Fund']
--------------------------------------------------
2. Total Unique Categories (2):
['Equity' 'Debt']
--------------------------------------------------
3. Total Unique Sub-categories (12):
['Large Cap' 'Small Cap' 'Gilt' 'Mid Cap' 'Short Duration' 'Value'
 'Liquid' 'Index/ETF' 'Flexi Cap' 'Index' 'Large & Mid Cap' 'ELSS']
--------------------------------------------------
4. Total Unique Risk Categories (5):
['Moderate' 'Very High' 'Low' 'High' 'Moderately High']
--------------------------------------------------


In [ ]:
import pandas as pd

# 1. Load both CSV datasets
df_master = pd.read_csv("../data/raw/01_fund_master.csv")
df_nav = pd.read_csv("../data/raw/02_nav_history.csv")

# 2. Extract unique AMFI codes as sets for comparison
master_codes = set(df_master['amfi_code'].unique())
nav_codes = set(df_nav['amfi_code'].unique())

print("==== DATA QUALITY REPORT ====\n")
print(f"Total unique codes in Fund Master: {len(master_codes)}")
print(f"Total unique codes in NAV History: {len(nav_codes)}")

# 3. Check if all master codes exist in the NAV history
missing_in_nav = master_codes - nav_codes

if len(missing_in_nav) == 0:
    print("\n[SUCCESS] Data quality check passed! All AMFI codes from Fund Master exist in NAV History.")
else:
    print(f"\n[WARNING] Data quality mismatch! These {len(missing_in_nav)} codes are missing in NAV History:")
    print(missing_in_nav)



==== DATA QUALITY REPORT ====

Total unique codes in Fund Master: 40
Total unique codes in NAV History: 40

[SUCCESS] Data quality check passed! All AMFI codes from Fund Master exist in NAV History.


In [ ]:
import pandas as pd
df_nav = pd.read_csv("../data/raw/02_nav_history.csv")
print(df_nav.columns)

Index(['amfi_code', 'date', 'nav'], dtype='object')


In [ ]:
import pandas as pd
import os

# 1. Load the raw NAV history dataset
df_nav = pd.read_csv("../data/raw/02_nav_history.csv")

# 2. Parse date column to datetime format (Using 'date' instead of 'nav_date')
df_nav['date'] = pd.to_datetime(df_nav['date'])

# 3. Sort by amfi_code and date to ensure chronological order
df_nav = df_nav.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)

# 4. Remove duplicate rows if any
df_nav = df_nav.drop_duplicates(subset=['amfi_code', 'date'])

# 5. Forward-fill missing NAV values for holidays/weekends per fund
df_nav['nav'] = df_nav.groupby('amfi_code')['nav'].ffill()

# 6. Validate that all NAV values are greater than 0
invalid_nav_count = (df_nav['nav'] <= 0).sum()

print("==== NAV CLEANING SUMMARY ====\n")
print(f"Total rows after cleaning: {len(df_nav)}")
print(f"Invalid NAV rows found (<= 0): {invalid_nav_count}")

# 7. Save the cleaned data to data/processed/ folder
os.makedirs("../data/processed", exist_ok=True)
df_nav.to_csv("../data/processed/clean_nav.csv", index=False)
print("\n[SUCCESS] Cleaned NAV data saved to data/processed/clean_nav.csv")

==== NAV CLEANING SUMMARY ====

Total rows after cleaning: 46000
Invalid NAV rows found (<= 0): 0

[SUCCESS] Cleaned NAV data saved to data/processed/clean_nav.csv


In [ ]:
import pandas as pd
import numpy as np
import os

processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

print("==== STARTING MASTER DATA CLEANING ====\n")

# --- 1. Clean investor_transactions.csv ---
try:
    df_tx = pd.read_csv("../data/raw/08_investor_transactions.csv")
    df_tx['transaction_type'] = df_tx['transaction_type'].str.strip().str.upper()
    
    # Using the correct column name 'amount_inr'
    df_tx = df_tx[df_tx['amount_inr'] > 0]
    
    if 'kyc_status' in df_tx.columns:
        df_tx['kyc_status'] = df_tx['kyc_status'].str.strip().str.upper()
        
    df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])
    df_tx.to_csv(f"{processed_dir}/clean_transactions.csv", index=False)
    print("[SUCCESS] clean_transactions.csv saved successfully!")
except Exception as e:
    print(f"[ERROR] Cleaning transactions: {e}")

# --- 2. Clean scheme_performance.csv ---
try:
    df_perf = pd.read_csv("../data/raw/07_scheme_performance.csv")
    for col in ['return_1yr', 'return_3yr', 'return_5yr']:
        if col in df_perf.columns:
            df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')
    if 'sharpe_ratio' in df_perf.columns:
        df_perf['sharpe_ratio'] = pd.to_numeric(df_perf['sharpe_ratio'], errors='coerce')
        df_perf['is_negative_sharpe'] = df_perf['sharpe_ratio'] < 0
    df_perf.to_csv(f"{processed_dir}/clean_performance.csv", index=False)
    print("[SUCCESS] clean_performance.csv saved.")
except Exception as e:
    print(f"[ERROR] Cleaning performance: {e}")

# --- 3. Clean and copy remaining master files ---
remaining_files = {
    "01_fund_master.csv": "clean_fund_master.csv",
    "03_aum_by_fund_house.csv": "clean_aum.csv",
    "04_monthly_sip_inflows.csv": "clean_sip_inflows.csv",
    "05_category_inflows.csv": "clean_category_inflows.csv",
    "06_industry_folio_count.csv": "clean_folio_count.csv",
    "09_portfolio_holdings.csv": "clean_portfolio.csv",
    "10_benchmark_indices.csv": "clean_benchmarks.csv"
}

for raw_file, clean_name in remaining_files.items():
    try:
        df = pd.read_csv(f"../data/raw/{raw_file}")
        df = df.drop_duplicates()
        df.to_csv(f"{processed_dir}/{clean_name}", index=False)
        print(f"[SUCCESS] {clean_name} saved.")
    except Exception as e:
        print(f"[ERROR] Processing {raw_file}: {e}")

print("\n==== ALL DATASETS CLEANED AND SAVED SUCCESSFULLY ====")

==== STARTING MASTER DATA CLEANING ====

[SUCCESS] clean_transactions.csv saved successfully!
[SUCCESS] clean_performance.csv saved.
[SUCCESS] clean_fund_master.csv saved.
[SUCCESS] clean_aum.csv saved.
[SUCCESS] clean_sip_inflows.csv saved.
[SUCCESS] clean_category_inflows.csv saved.
[SUCCESS] clean_folio_count.csv saved.
[SUCCESS] clean_portfolio.csv saved.
[SUCCESS] clean_benchmarks.csv saved.

==== ALL DATASETS CLEANED AND SAVED SUCCESSFULLY ====


In [11]:
import pandas as pd
import numpy as np
import os

processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

print("==== STARTING MASTER DATA CLEANING FROM SCRATCH ====\n")

# --- 1. Clean investor_transactions.csv ---
try:
    df_tx = pd.read_csv("../data/raw/08_investor_transactions.csv")
    df_tx['transaction_type'] = df_tx['transaction_type'].str.strip().str.upper()
    
    # Using the correct column name 'amount_inr'
    df_tx = df_tx[df_tx['amount_inr'] > 0]
    
    if 'kyc_status' in df_tx.columns:
        df_tx['kyc_status'] = df_tx['kyc_status'].str.strip().str.upper()
        
    df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])
    df_tx.to_csv(f"{processed_dir}/clean_transactions.csv", index=False)
    print("[SUCCESS] clean_transactions.csv saved successfully!")
except Exception as e:
    print(f"[ERROR] Cleaning transactions: {e}")

# --- 2. Clean scheme_performance.csv ---
try:
    df_perf = pd.read_csv("../data/raw/07_scheme_performance.csv")
    for col in ['return_1yr', 'return_3yr', 'return_5yr']:
        if col in df_perf.columns:
            df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')
    if 'sharpe_ratio' in df_perf.columns:
        df_perf['sharpe_ratio'] = pd.to_numeric(df_perf['sharpe_ratio'], errors='coerce')
        df_perf['is_negative_sharpe'] = df_perf['sharpe_ratio'] < 0
    df_perf.to_csv(f"{processed_dir}/clean_performance.csv", index=False)
    print("[SUCCESS] clean_performance.csv saved.")
except Exception as e:
    print(f"[ERROR] Cleaning performance: {e}")

# --- 3. Clean and copy remaining master files ---
remaining_files = {
    "01_fund_master.csv": "clean_fund_master.csv",
    "03_aum_by_fund_house.csv": "clean_aum.csv",
    "04_monthly_sip_inflows.csv": "clean_sip_inflows.csv",
    "05_category_inflows.csv": "clean_category_inflows.csv",
    "06_industry_folio_count.csv": "clean_folio_count.csv",
    "09_portfolio_holdings.csv": "clean_portfolio.csv",
    "10_benchmark_indices.csv": "clean_benchmarks.csv"
}

for raw_file, clean_name in remaining_files.items():
    try:
        df = pd.read_csv(f"../data/raw/{raw_file}")
        df = df.drop_duplicates()
        df.to_csv(f"{processed_dir}/{clean_name}", index=False)
        print(f"[SUCCESS] {clean_name} saved.")
    except Exception as e:
        print(f"[ERROR] Processing {raw_file}: {e}")

print("\n==== ALL DATASETS CLEANED AND SAVED SUCCESSFULLY ====")

==== STARTING MASTER DATA CLEANING FROM SCRATCH ====

[SUCCESS] clean_transactions.csv saved successfully!
[SUCCESS] clean_performance.csv saved.
[SUCCESS] clean_fund_master.csv saved.
[SUCCESS] clean_aum.csv saved.
[SUCCESS] clean_sip_inflows.csv saved.
[SUCCESS] clean_category_inflows.csv saved.
[SUCCESS] clean_folio_count.csv saved.
[SUCCESS] clean_portfolio.csv saved.
[SUCCESS] clean_benchmarks.csv saved.

==== ALL DATASETS CLEANED AND SAVED SUCCESSFULLY ====


In [12]:
import pandas as pd
from sqlalchemy import create_engine
import os

# 1. Create SQLite database engine
# This will automatically create 'bluestock_mf.db' in your project root folder
engine = create_engine('sqlite:///../bluestock_mf.db')

processed_dir = "../data/processed"

print("==== STARTING DATABASE LOADING (TASK 5) ====\n")

# Mapping our cleaned CSV files to their respective SQLite tables
tables_to_load = {
    "clean_fund_master.csv": "dim_fund",
    "clean_nav.csv": "fact_nav",
    "clean_transactions.csv": "fact_transactions",
    "clean_performance.csv": "fact_performance"
}

# Load each core table
for csv_file, table_name in tables_to_load.items():
    csv_path = f"{processed_dir}/{csv_file}"
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            
            # Load data into SQLite table (if table exists, append data)
            df.to_sql(table_name, engine, if_exists='replace', index=False)
            print(f"[SUCCESS] Loaded {csv_file} into table '{table_name}' ({len(df)} rows).")
        except Exception as e:
            print(f"[ERROR] Failed to load {csv_file}: {e}")
    else:
        print(f"[WARNING] File not found: {csv_path}")

# Load remaining supporting analysis CSVs into the DB for Day 3/4 queries
remaining_tables = {
    "clean_aum.csv": "fact_aum",
    "clean_sip_inflows.csv": "fact_sip_industry",
    "clean_category_inflows.csv": "fact_category_inflows",
    "clean_folio_count.csv": "fact_folio_count",
    "clean_portfolio.csv": "fact_portfolio",
    "clean_benchmarks.csv": "fact_benchmarks"
}

print("\n--- Loading Supporting Tables ---")
for csv_file, table_name in remaining_tables.items():
    csv_path = f"{processed_dir}/{csv_file}"
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            df.to_sql(table_name, engine, if_exists='replace', index=False)
            print(f"[SUCCESS] Loaded {csv_file} into table '{table_name}' ({len(df)} rows).")
        except Exception as e:
            print(f"[ERROR] Failed to load {csv_file}: {e}")

print("\n==== ALL DATASETS LOADED INTO BLUESTOCK_MF.DB SUCCESSFULLY ====")

==== STARTING DATABASE LOADING (TASK 5) ====

[SUCCESS] Loaded clean_fund_master.csv into table 'dim_fund' (40 rows).
[SUCCESS] Loaded clean_nav.csv into table 'fact_nav' (46000 rows).
[SUCCESS] Loaded clean_transactions.csv into table 'fact_transactions' (32778 rows).
[SUCCESS] Loaded clean_performance.csv into table 'fact_performance' (40 rows).

--- Loading Supporting Tables ---
[SUCCESS] Loaded clean_aum.csv into table 'fact_aum' (90 rows).
[SUCCESS] Loaded clean_sip_inflows.csv into table 'fact_sip_industry' (48 rows).
[SUCCESS] Loaded clean_category_inflows.csv into table 'fact_category_inflows' (144 rows).
[SUCCESS] Loaded clean_folio_count.csv into table 'fact_folio_count' (21 rows).
[SUCCESS] Loaded clean_portfolio.csv into table 'fact_portfolio' (322 rows).
[SUCCESS] Loaded clean_benchmarks.csv into table 'fact_benchmarks' (8050 rows).

==== ALL DATASETS LOADED INTO BLUESTOCK_MF.DB SUCCESSFULLY ====
